# Nüks (cured vs recurrence) modeli — 5-fold çapraz doğrulama

**Google Colab** üzerinde çalışır. Bilgisayarına hiçbir şey kurman gerekmez.

Hücreleri **sırayla** çalıştır: soldaki ▶ düğmesine bas, bitmesini bekle, sonrakine geç.
Bir hücre hata verirse durup bana çıktıyı gönder — sonraki hücrelere geçme.

---
**Başlamadan:** Runtime → Change runtime type → **T4 GPU** → Save.


## 1. Kodu indir

Bu hücre `recurrence_cv.py` dosyasını GitHub'dan çeker.


In [ ]:
!wget -q -O recurrence_cv.py https://raw.githubusercontent.com/hakanyalman96-cell/MultiQRSNet-Recurrence/main/recurrence_cv.py

import os
if os.path.exists('recurrence_cv.py') and os.path.getsize('recurrence_cv.py') > 5000:
    print('Kod indirildi:', os.path.getsize('recurrence_cv.py'), 'bayt')
else:
    print('HATA: indirilemedi. Repo public mi kontrol et.')


## 2. Ortam kontrolü


In [ ]:
import torch, sklearn
print('PyTorch     :', torch.__version__)
print('scikit-learn:', sklearn.__version__)
from sklearn.model_selection import StratifiedGroupKFold
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'yok (CPU ile calisir, yavas)')


## 3. Google Drive'ı bağla

Link çıkacak: hesabını seç, izin ver.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Veri klasörünü bul


In [ ]:
import os

hits = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train-test-folder' in dirs:
        hits.append(os.path.join(root, 'train-test-folder'))
    if len(root.split(os.sep)) > 9:
        dirs[:] = []

if hits:
    for h in hits: print('bulundu:', h)
    DATA = hits[0]
    print('\nkullanilacak ->', DATA)
else:
    DATA = ''
    print('Bulunamadi.')
    print('Sol taraftaki klasor ikonundan yolu bul, asagiya elle yaz:')
    print("DATA = '/content/drive/MyDrive/.../train-test-folder'")


Bulunamadıysa yolu elle yaz (satır başındaki `#` işaretini kaldır):


In [ ]:
# DATA = '/content/drive/MyDrive/BURAYA_YOL/train-test-folder'
print('DATA =', DATA)


## 5. Klasör yapısı doğru mu?


In [ ]:
for site in ['LVOT','RVOT','RLVOT']:
    p = os.path.join(DATA, site)
    if os.path.isdir(p):
        subs = sorted(os.listdir(p))
        n = sum(len(fs) for _,_,fs in os.walk(p))
        print(f'{site:6} -> {subs}  ({n} dosya)')
    else:
        print(f'{site:6} -> YOK')


## 6. Dosya formatı kontrolü

Kod, CARTO metin dosyalarında **3. satırın başlık satırı** olduğunu ve
derivasyon isimlerinin `V1(22)`, `I(110)` gibi yazıldığını varsayıyor.
Bu hücre gerçek dosyalarından birine bakıp bu varsayımı doğrular.


In [ ]:
import glob

f = None
for pat in [os.path.join(DATA,'*','*','**','*.txt'), os.path.join(DATA,'*','*','*.txt')]:
    hits = glob.glob(pat, recursive=True)
    if hits: f = hits[0]; break

if not f:
    print('Hic .txt dosyasi bulunamadi - DATA yolunu kontrol et.')
else:
    print('ornek dosya:', f, '\n')
    lines = open(f).readlines()
    for i in range(min(4, len(lines))):
        print(f'satir {i}: {lines[i][:160].rstrip()}')
    print()
    need = ['V1(22)','V2(23)','V3(24)','V4(25)','V5(26)','V6(27)',
            'I(110)','II(111)','III(112)','aVL(171)','aVR(172)','aVF(173)']
    hdr = lines[2].split() if len(lines) > 2 else []
    missing = [c for c in need if c not in hdr]
    if not missing:
        print('OK: 12 derivasyonun hepsi 3. satirda bulundu. Devam edebilirsin.')
    else:
        print('EKSIK sutunlar:', missing)
        print('\n3. satirdaki gercek sutun isimleri:')
        print(hdr[:40])
        print('\nBu ciktiyi bana gonder, parser i formatina gore duzeltirim.')


## 7. ÖNEMLİ — hasta gruplaması doğru mu?

**Bu adımı atlama.** Leakage koruması, hangi kayıtların aynı hastaya ait olduğunun
doğru çözülmesine bağlı.

Çıktıda `recordings/patient` satırına bak:
- **1'den büyükse** gruplama çalışıyor → 8. hücreye geç.
- **Tam olarak 1 ise** her kayıt ayrı hasta sayılmış demektir → koruma etkisiz,
  aşağıdaki alternatifleri dene.


In [ ]:
!python recurrence_cv.py --data "$DATA" --sanity-check


### Hasta ID'si yanlış çözüldüyse
Sırayla dene, hangisi doğru gruplarsa onu kullan (`#` işaretini kaldır):


In [ ]:
# Hasta = dosyanin icinde bulundugu klasor adi
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id parent

# Hasta = dosya adinin ilk _ veya - isaretine kadarki kismi
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id stem

# Kendi kalibin (ornek: dosya adi 'HASTA042_kayit3.txt')
# !python recurrence_cv.py --data "$DATA" --sanity-check --patient-id regex --patient-pattern '(HASTA\d+)'


## 8. Eğitim

7. hücrenin çıktısı doğruysa burayı çalıştır. GPU ile birkaç dakika sürer.


In [ ]:
!python recurrence_cv.py --data "$DATA" --out results_recurrence --epochs 60 --seed 42


## 9. Sonuç tabloları

Bunlar doğrudan Supplementary Tablo S2 ve S3'e girer.


In [ ]:
import pandas as pd, json

print('=== Tablo S2: fold kompozisyonu ===')
display(pd.read_csv('results_recurrence/fold_composition.csv'))

print('=== Tablo S3: fold bazinda metrikler ===')
display(pd.read_csv('results_recurrence/fold_metrics.csv').round(4))

s = json.load(open('results_recurrence/summary.json'))
print('=== Fold ortalamasi (%95 GA) ===')
for k, v in s['per_fold_mean_95ci'].items():
    print(f"  {k:<10} {v['mean']:.4f}  (95% GA {v['lo']:.4f} - {v['hi']:.4f})")

print('\n=== HASTA DUZEYI — yazida birincil olarak bunu raporla ===')
for k, v in s['pooled_oof_patient_level'].items():
    print(f'  {k:<10} {v}')
ci = s['patient_level_bootstrap_ci']
print(f"  AUC    %95 GA: {ci['auc'][0]:.4f} - {ci['auc'][1]:.4f}")
print(f"  PR-AUC %95 GA: {ci['pr_auc'][0]:.4f} - {ci['pr_auc'][1]:.4f}")

prev = s['n_recurrence_patients'] / s['n_patients']
print(f"\nNuks prevalansi (PR-AUC temel cizgisi): {prev:.4f}")
print('PR-AUC bunun belirgin uzerinde degilse model klinik bilgi katmiyordur.')


## 10. MDI ablasyonu (hakem sorusuna cevap)

"SHAP'a göre MDI katkısı zayıf, neden tutuyorsunuz?" sorusunun cevabı bu.


In [ ]:
!python recurrence_cv.py --data "$DATA" --out results_no_mdi --epochs 60 --seed 42 --no-mdi

import json
a = json.load(open('results_recurrence/summary.json'))['pooled_oof_patient_level']
b = json.load(open('results_no_mdi/summary.json'))['pooled_oof_patient_level']
print(f"{'metrik':<10}{'MDI ile':>10}{'MDI siz':>10}")
for k in ['accuracy','f1','auc','pr_auc']:
    print(f'{k:<10}{a[k]:>10.4f}{b[k]:>10.4f}')


## 11. Farklı tohumlarla tekrar

Bu boyutta veride tek çalıştırma yanıltıcı olabilir; yayılımı raporlamak daha dürüst.


In [ ]:
import json, pandas as pd
rows = []
for seed in [1, 2, 3, 42, 2024]:
    !python recurrence_cv.py --data "$DATA" --out results_seed$seed --epochs 60 --seed $seed > /dev/null 2>&1
    m = json.load(open(f'results_seed{seed}/summary.json'))['pooled_oof_patient_level']
    rows.append({'seed': seed, **{k: round(m[k], 4) for k in ['accuracy','f1','auc','pr_auc']}})

df = pd.DataFrame(rows)
display(df)
print('Tohumlar arasi ortalama +/- SS:')
for k in ['accuracy','f1','auc','pr_auc']:
    print(f'  {k:<10}{df[k].mean():.4f} +/- {df[k].std():.4f}')


## 12. Sonuçları indir


In [ ]:
!zip -qr results.zip results_* 
from google.colab import files
files.download('results.zip')
